# EDA and data cleaning for the PPR csv data

## Imports

In [1]:
import pandas as pd
from pathlib import Path
import re
from deepparse.parser import AddressParser

/workspaces/ireland-property-analysis/.venv/lib/python3.12/site-packages/pymagnitudelight/framework/repoze/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


## read raw data

In [3]:
# Path to the file we downloaded in Iteration 1
raw_data_path = Path("../data/raw/PPR-ALL.csv")

# Note: The PPR file uses 'latin-1' or 'cp1252' encoding, not standard 'utf-8'
raw_df = pd.read_csv(raw_data_path, encoding='cp1252')

/tmp/ipykernel_1742/2864090230.py:5: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(raw_data_path, encoding='cp1252')


## read a sample from the raw data

In [4]:
print(f"Dataset contains {len(raw_df):,} rows.")
raw_df.head()

Dataset contains 768,790 rows.


,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description
0,01/01/2010,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN,"€343,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
1,03/01/2010,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN,"€185,000.00",No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...
2,04/01/2010,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN,"€438,500.00",No,No,Second-Hand Dwelling house /Apartment,NaN
3,04/01/2010,"1 The Haven, Mornington",Meath,NaN,"€400,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
4,04/01/2010,"11 Melville Heights, Kilkenny",Kilkenny,NaN,"€160,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN


## create deep copy of the raw data

In [5]:
df = raw_df.copy(deep=True)

## clean columns names

In [6]:
df.columns = (
    df.columns
    .str.replace(r'[^\w\s]', '', regex=True) # Removes (€) and other symbols
    .str.strip()                             # Removes leading/trailing spaces
    .str.replace(' ', '_')                   # Replaces middle spaces with underscores
    .str.lower()                             # Makes everything lowercase
)
print(df.columns)

Index(['date_of_sale_ddmmyyyy', 'address', 'county', 'eircode', 'price',
       'not_full_market_price', 'vat_exclusive', 'description_of_property',
       'property_size_description'],
      dtype='object')


## Price cleaning and enrich

### clean price column from non numeric symbols

#### create a function for cleaning the price string

In [7]:
def clean_currency(price_str):
    if pd.isna(price_str):
        return None
    # remove everything that isn't a digit or descimal point
    clean_str = re.sub(r'[^\d.]','',str(price_str))
    # convert clean_str to floot which handles the .00 then to int
    try:
        return int(float(clean_str))
    except ValueError:
        return None

#### use the function to clean

In [8]:
df['price_clean'] = df['price'].apply(clean_currency)
print(f"Average Price in euros: {df['price_clean'].mean()}")

Average Price in euros: 313460.9311723618


### create boolean column for vat exclusive

In [9]:
df['is_vat_exclusive'] = df['vat_exclusive'].str.contains('Yes',case=False, na=False)

## create a smaller sample dataset for fast debugginf and development

#### new folder so raw data are safe

In [10]:
# Define your paths
processed_dir = Path("../data/processed")

# Create the folder if it doesn't exist
processed_dir.mkdir(parents=True, exist_ok=True)

#### filter county dublin only

In [ ]:
# 1. Filter for Dublin (Case-insensitive just in case)
dublin_df = df[df['county'].str.contains('Dublin', case=False, na=False)].copy()

# 2. Save to new folder
output_path = processed_dir / "dublin_sample.csv"
dublin_df.to_csv(output_path, index=False)

print(f"Success! Saved {len(dublin_df):,} rows to {output_path}")

### use deepparse on the dublin sample dataset

In [11]:
df_dublin = pd.read_csv("../data/processed/dublin_sample.csv")
#Initialize the parser (the first time takes a moment to download the model)
# We use 'fasttext' because it's lightweight and runs great on a Mac
address_parser = AddressParser(model_type="bpemb", device="cpu")
#Test on a single messy Dublin address
test_address = df_dublin['address'].iloc[0]
parsed = address_parser(test_address)

print(f"Original: {test_address}")
print(f"Parsed: {parsed}")


/tmp/ipykernel_1742/1669742642.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dublin = pd.read_csv("../data/processed/dublin_sample.csv")
/workspaces/ireland-property-analysis/.venv/lib/python3.12/site-packages/deepparse/download_tools.py:92: UserWarning: The offline parameter is set to False, so if a new pre-trained `bpemb` model is available it will automatically be downloaded.
  warnings.warn(


Loading the embeddings model
downloading https://bpemb.h-its.org/multi/multi/multi.wiki.bpe.vs100000.model


100%|██████████| 1965223/1965223 [00:00<00:00, 6842142.45B/s]


downloading https://bpemb.h-its.org/multi/multi/multi.wiki.bpe.vs100000.d300.w2v.bin.tar.gz


100%|██████████| 112202964/112202964 [00:02<00:00, 44635328.11B/s]


Original: 5 Braemor Drive, Churchtown, Co.Dublin
Parsed: The unparsed address is '5 Braemor Drive, Churchtown, Co.Dublin' and the parsed address is '('5', 'StreetNumber') ('braemor', 'StreetName') ('drive', 'StreetName') ('churchtown', 'StreetName') ('co.dublin', 'Municipality')'
